<small><font color=gray>Notebook author: <a href="https://www.linkedin.com/in/olegmelnikov/" target="_blank">Oleg Melnikov</a> ©2021 onwards</font></small><hr style="margin:0;background-color:silver">

**<font size=6>🦠Bacteria</font>**. [**Instructions**](https://colab.research.google.com/drive/1riOGrE_Fv-yfIbM5V4pgJx4DWcd92cZr#scrollTo=ITaPDPIQEgXV) for running Colabs.

<details>
  <summary><small>Sharing consent: <mark>[ X ]</mark></summary>
  <div>
We consent to sharing our Colab (after the assignment ends) with other students/instructors for educational purposes. We understand that sharing is <b>optional</b> and this decision will not affect our grade in any way. <font color=gray><i>
Instructions: If ok with sharing your Colab for educational purposes, leave "X" in the check box.</i></font></small></div>

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
# OK to enable, if your kaggle.json is stored in Google Drive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!mkdir -p ~/.kaggle                               # .kaggle folder must contain kaggle.json for kaggle executable to properly authenticate you to Kaggle.com
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json >>log  # First, download kaggle.json from kaggle.com (in Account page) and place it in the root of mounted Google Drive
!cp kaggle.json ~/.kaggle/kaggle.json >> log      # Alternative location of kaggle.json (without a connection to Google Drive)
!chmod 600 ~/.kaggle/kaggle.json                  # give only the owner full read/write access to kaggle.json
!kaggle config set -n competition -v 260223-bacteria # set the competition context for the next few kaggle API calls. !kaggle config view - shows current settings
!kaggle competitions download >> log              # download competition dataset as a zip file
!unzip -o *.zip >> log                            # Kaggle dataset is copied as a single file and needs to be unzipped.
!kaggle competitions leaderboard --show           # print public leaderboard

cp: cannot stat 'kaggle.json': No such file or directory
- competition is now set to: 260223-bacteria
Using competition: 260223-bacteria
  teamId  teamName     submissionDate              score         
--------  -----------  --------------------------  ------------  
15295157  Andrew Safe  2026-02-24 05:20:01.383000  0.9620807103  
15285697  🦠Baseline    2026-02-22 17:24:12.360000  0.7736013068  


In [ ]:
%%time
%%capture log
%reset -f
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all"
import numpy as np, pandas as pd, time, matplotlib.pyplot as plt, seaborn as sns, os, tensorflow as tf, tensorflow.keras as keras
from keras.layers import Flatten, Dense
from keras.models import Sequential
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
os.environ['TF_DETERMINISTIC_OPS'] = '1'; os.environ['TF_CUDNN_DETERMINISTIC'] = '1'; # allows seeding RNG on GPU
ToCSV = lambda df, fname: df.round(2).to_csv(f'{fname}.csv', index_label='id') # rounds values to 2 decimals

class Timer():
  def __init__(self, lim:'RunTimeLimit'=60): self.t0, self.lim, _ = time.time(), lim, print(f'⏳ started. You have {lim} sec. Good luck!')
  def ShowTime(self):
    msg = f'Runtime is {time.time()-self.t0:.0f} sec'
    print(f'\033[91m\033[1m' + msg + f' > {self.lim} sec limit!!!\033[0m' if (time.time()-self.t0-1) > self.lim else msg)

np.set_printoptions(linewidth=100, precision=2, edgeitems=2, suppress=True)
pd.set_option('display.max_columns', 20, 'display.precision', 2, 'display.max_rows', 4)

CPU times: user 5.11 s, sys: 783 ms, total: 5.89 s
Wall time: 5.79 s


In [ ]:
df = pd.read_csv('XY_Bacteria.csv'); df

,A0T0G0C10,A0T0G1C9,A0T0G2C8,A0T0G3C7,A0T0G4C6,A0T0G5C5,A0T0G6C4,A0T0G7C3,A0T0G8C2,A0T0G9C1,...,A8T0G1C1,A8T0G2C0,A8T1G0C1,A8T1G1C0,A8T2G0C0,A9T0G0C1,A9T0G1C0,A9T1G0C0,A10T0G0C0,y
0,10,32,41,39,77,122,55,81,58,31,...,38,88,80,20,36,31,32,32,10,NaN
1,10,31,52,165,225,221,150,143,48,31,...,111,125,143,159,69,45,32,71,10,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99998,10,31,66,107,142,155,142,107,66,31,...,93,66,93,93,66,31,31,31,10,4.0
99999,10,32,41,21,45,89,45,60,67,31,...,97,82,74,80,52,31,32,7,10,2.0


In [ ]:
df.info()   # observe datatypes and any missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Columns: 287 entries, A0T0G0C10 to y
dtypes: float64(1), int64(286)
memory usage: 219.0 MB


In [ ]:
vX = df.query('y!=y').drop('y', axis=1)    # slice a test sample
tXY = df.query('y==y')                     # slice training sample
tX, tY = tXY.drop('y', axis=1), tXY.y.astype(int)      # split into training I/O
vX  # test inputs
tX  # train inputs
print(tY.tolist()[:50]) # train outputs

,A0T0G0C10,A0T0G1C9,A0T0G2C8,A0T0G3C7,A0T0G4C6,A0T0G5C5,A0T0G6C4,A0T0G7C3,A0T0G8C2,A0T0G9C1,...,A8T0G0C2,A8T0G1C1,A8T0G2C0,A8T1G0C1,A8T1G1C0,A8T2G0C0,A9T0G0C1,A9T0G1C0,A9T1G0C0,A10T0G0C0
0,10,32,41,39,77,122,55,81,58,31,...,27,38,88,80,20,36,31,32,32,10
1,10,31,52,165,225,221,150,143,48,31,...,76,111,125,143,159,69,45,32,71,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9998,10,31,66,107,142,156,143,107,66,31,...,420,296,303,421,302,306,31,31,31,10
9999,10,31,66,107,142,155,142,108,66,31,...,66,429,307,93,93,66,31,31,31,10


,A0T0G0C10,A0T0G1C9,A0T0G2C8,A0T0G3C7,A0T0G4C6,A0T0G5C5,A0T0G6C4,A0T0G7C3,A0T0G8C2,A0T0G9C1,...,A8T0G0C2,A8T0G1C1,A8T0G2C0,A8T1G0C1,A8T1G1C0,A8T2G0C0,A9T0G0C1,A9T0G1C0,A9T1G0C0,A10T0G0C0
10000,10,7,48,74,119,110,127,86,66,7,...,223,319,233,361,439,368,179,174,199,10
10001,10,31,66,107,142,156,142,108,66,31,...,66,539,302,93,424,434,31,313,31,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99998,10,31,66,107,142,155,142,107,66,31,...,66,93,66,93,93,66,31,31,31,10
99999,10,32,41,21,45,89,45,60,67,31,...,52,97,82,74,80,52,31,32,7,10


[3, 3, 4, 0, 4, 3, 4, 4, 3, 3, 2, 2, 4, 2, 2, 4, 4, 2, 2, 0, 2, 3, 3, 3, 4, 3, 3, 4, 3, 3, 3, 4, 4, 2, 3, 3, 2, 0, 1, 4, 4, 2, 3, 4, 3, 0, 3, 2, 2, 0]


In [ ]:
pd.set_option('display.max_rows', 10)
# tY.value_counts()
df['y'].value_counts(normalize=True)

,proportion
y,
4.0,0.30
2.0,0.30
3.0,0.30
1.0,0.05
0.0,0.05


In [ ]:
tmr = Timer()

⏳ started. You have 60 sec. Good luck!


<hr color=green size=40>

<strong><font color=green size=5>⏳<span title="Timed Green Playground">TGP</span> - for your code, docs, timer!</font></strong>

<font color=green>
<details>
  <summary>Instructions</summary>
  <div>Students: Keep all your definitions, code, documentation in <b>TGP</b> (timed green playground). Modifying any code outside of TGP or exxceeding RTL (runtime limit, <code>Timer().lim</code>) incurs penalties - see <a href=https://drive.google.com/drive/folders/10_OAoFTdUg71Z0Op_ebxIxcNfQWByakT?usp=drive_link>Grading Rubric for Kaggle Colabs</a> Google Sheet.
</div> </details>
</font>

<font color=green><h3><b>$\alpha$. Split observations into train and test sets</b><h3>

In [ ]:
%%time
# PREPROCESSING: StandardScaler (fast, no PCA to save time)

tf.random.set_seed(0)
Init = keras.initializers.RandomNormal(seed=0)
np.random.seed(0)

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

tX_raw = tXY.drop('y', axis=1).values.astype('float32')
tY_all = tXY.y.values.astype(int)
vX_raw = vX.values.astype('float32')

# --- StandardScaler ---
scaler = StandardScaler()
tX_sc = scaler.fit_transform(tX_raw)
vX_sc = scaler.transform(vX_raw)

# --- Class weights for imbalance (classes 0,1 are rare ~2-3%) ---
cw = compute_class_weight('balanced', classes=np.unique(tY_all), y=tY_all)
class_weights = dict(enumerate(cw))

# --- Train/val split ---
tX_in, vX_val, tY_in, vY_val = train_test_split(
    tX_sc, tY_all, test_size=0.15, random_state=0, stratify=tY_all)

print(f"Train: {tX_in.shape}, Val: {vX_val.shape}, Classes: {np.unique(tY_all)}")


Train: (76500, 286), Val: (13500, 286), Classes: [0 1 2 3 4]
CPU times: user 722 ms, sys: 123 ms, total: 845 ms
Wall time: 874 ms


<font color=green><h3><b>$\beta$. Build and fit a model</b><h3>

In [ ]:
%%time
# MODEL: Lean DNN — fast training within 60s [SP25 rank-2, FA24 rank-2]
from keras.layers import BatchNormalization, Dropout

m = Sequential([
    Dense(128, activation='relu', input_shape=[tX_in.shape[1]],
          kernel_initializer='he_normal', name='h1'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_initializer='he_normal', name='h2'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(len(np.unique(tY_all)), activation='softmax', name='out')
])

m.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Nadam(learning_rate=0.002),
    metrics=['accuracy']
)

hist = m.fit(
    tX_in, tY_in,
    epochs=10,
    validation_data=(vX_val, vY_val),
    batch_size=512,               # large batch = fewer steps = faster
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, min_lr=1e-5)
    ],
    verbose=1
)

from sklearn.metrics import f1_score
val_pred = np.argmax(m.predict(vX_val, verbose=0), axis=-1)
print(f"\nVal macro F1: {f1_score(vY_val, val_pred, average='macro'):.4f}")
print(f"Val accuracy: {accuracy_score(vY_val, val_pred):.4f}")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.7088 - loss: 0.8550 - val_accuracy: 0.9379 - val_loss: 0.1651 - learning_rate: 0.0020
Epoch 2/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9116 - loss: 0.2547 - val_accuracy: 0.9547 - val_loss: 0.1211 - learning_rate: 0.0020
Epoch 3/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9365 - loss: 0.1906 - val_accuracy: 0.9614 - val_loss: 0.1030 - learning_rate: 0.0020
Epoch 4/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9481 - loss: 0.1581 - val_accuracy: 0.9327 - val_loss: 0.2216 - learning_rate: 0.0020
Epoch 5/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9522 - loss: 0.1431 - val_accuracy: 0.9734 - val_loss: 0.0747 - learning_rate: 0.0010
Epoch 6/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9612 - loss: 0.1175 - val_accuracy: 0.9747 - val_loss: 0.0715 - learning_rate: 0.0010
Epoch 7/10
150/150 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9647 - loss: 0.1087

<font color=green><h3><b>$\gamma$. Make predictions</b><h3>

In [ ]:
# === Generate predictions and save ===
pY = pd.DataFrame(
    np.argmax(m.predict(vX_sc, verbose=0), axis=-1),
    index=vX.index + 1, columns=['y']
)
ToCSV(pY.round(0).astype(int), '🦠Bacteria_Submission')
print('Observed  y:', list((tXY.y.value_counts(normalize=True).sort_index()).round(3)))
print('Predicted y:', list((pY['y'].value_counts(normalize=True).sort_index()).round(3)))
print(f"Saved: 🦠Bacteria_Submission.csv ({len(pY)} predictions)")


Observed  y: [0.048, 0.05, 0.302, 0.296, 0.303]
Predicted y: [0.054, 0.055, 0.298, 0.293, 0.299]
Saved: 🦠Bacteria_Submission.csv (10000 predictions)


<font color=green><h3><b>$\delta$. Idea Documentation</b></h3>
<details>
  <summary>Instructions</summary>
  <div>


1. **Audience**. Your peers who will learn from your Colab and ideas therein.
1. **Importance**. The ML/DL ideas are not entirely random, but are based on prior experience and systematized/organized experiments. We'd like students to share and learn from idea generation to idea experimentation process done in our class using tools learned thus far.
1. **Format**. Keep it concise/precise in consistent font/presentation. Include numbers/IDs to your References, such as [1] or [[Géron22]](https://scholar.google.com/scholar?cluster=498861685923226475), where these are defined in your References section below. This helps link your ideas/experiments to external ideas.
1. **Reproducibility**. Your description should contain reasonable details needed for reproducibility, i.e. describe the state of your modeling pipeline before the change is made, what is changed and how the idea was discovered, and what improvement it resulted in. Thus, peers can try this idea with an expectation of the value it brings. See examples below.
1. **Bonus** points for the exceptional/exemplary/educational documentation (see grading rubric).
****
1. **TODO**: Describe the key idea in your work in the following format (similar to a "micro publication"):
  1. **Title**. Give each idea a descriptive name (i.e. a micro abstract).
    1. Ex(ample). <i>"Thresholding carat feature outliers improves MAE by 3% on public LB"</i>
  1. **Idea Discovery**. What led you to this idea? Was it some [EDA](https://en.wikipedia.org/wiki/Exploratory_data_analysis), familiarity with this dataset or some of the features?
    1. Ex. <i>"We plotted all univariate distributions of variables and discovered that diamond carat had unreasonable (but rare) values below and above [0,10] interval, when plotted carat's histogram in the train and test sets, which contained 10 and 3 such outliers respectively. We decided to use 10 as a reasonable threshold because it is 99th percentile of carat values in the 20K baseline sample. See our histogram plot below [plot here]. "</i>
  1. **Finding's Importance**. Describe why you think the idea was important to proceed with.
    1. Ex. <i>"We use a linear model, the slope of which is sensitive to outliers on the periphery of the feature space domain. The fitted hyperplane slopes in the direction of the extreme training feature values thereby mapping a non-existent relation between carat size and diamond price, which is not expected to repeat in the test set. "</i>
  1. **Experiment Setup**.
  How did you set up experiments to test your idea? What resources were helpful? What metric did you select, why and what values did you observe?
    1. Ex. <i>"To alleviate the impact of the outlying feature values, we need to either remove observations with extreme values, or somehow cap them (to stay within the distribution of the other carat values) or use a model insensitive to outliers (such as robust regression). We learned 3 suitable methods for treating outliers in [ref]: ... [It'd be great to briefly describe each method] We tried each one on a Baseline model, while keeping the competition-required [MAE](https://en.wikipedia.org/wiki/Mean_absolute_error) metric. We tested each method locally on the seeded 50/50 split of the 20K training set sampled in baseline Colab."</i>
  1. **Results**. What was the result or metric improvement from implementing the experiment locally and/or on public LB?
    1. Ex. <i>"Baseline MAE was 539.1257546465 in public LB and 530 in local default experiment with 50/50 train-test split. When applied on the same-seed split, Methods 1,2,and 3 showed 1%, 2%, and 5% improvement on the test set. When uploaded to public LB, Method 3 showed a 3% improvement. So, we decided to keep method 3."</i>

</div> </details>
</font>


<font color=green><h4><b>Task 1. Preprocessing Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped in <b>preprocessing pipeline</b>. This may be about some feature engineering, tricky subsampling, clustering, dimension reduction, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

1. **Title**: StandardScaler with Class Weight Balancing
1. **Idea Discovery**: All top past teams used StandardScaler [SP25 rank-1,2,3; FA24 rank-2,3]. Class imbalance (classes 0,1 are ~2-3%) is addressed via `compute_class_weight('balanced')` which directly improves macro F1 for rare classes.
1. **Finding's Importance**: Raw k-mer count features have varying scales. StandardScaler ensures zero-mean, unit-variance for stable DNN training [10]. Class weights penalize rare-class misclassification proportionally, critical since metric is macro F1 where all classes count equally.
1. **Experiment Setup**: StandardScaler on all training data. Class weights computed via sklearn. 85/15 stratified train/val split.
1. **Results**: Scaling + class weights provide a strong foundation for the DNN, ensuring rare classes (0,1) are properly learned.


<font color=green><h4><b>Task 2. Modeling Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped with <b>model selection</b> in the format specified above. This may include tuning model parameters (perhaps a grid search with specific parameter range) or some other experiments, search/choice of the suitable model, experiments with postprocessing of model predictions, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

1. **Title**: Lean DNN with BatchNorm, Dropout, and Nadam
1. **Idea Discovery**: SP25 rank-2 (F1=0.9905) found depth outperforms width. FA24 rank-2 (F1=0.9933) used BatchNorm, Dropout, and Nadam. We use a 2-layer architecture (128→64) optimized for the 60s time limit with batch_size=512 and 10 epochs with EarlyStopping.
1. **Finding's Importance**: Baseline (1 layer, 50 units, 2 epochs) severely underfits. Deeper architecture with regularization captures complex k-mer patterns. Large batch size (512) reduces training steps per epoch. Nadam optimizer combines Adam with Nesterov momentum for faster convergence [14]. EarlyStopping halts when validation loss plateaus.
1. **Experiment Setup**: Dense(128)→BN→Dropout(0.3) → Dense(64)→BN→Dropout(0.2) → Softmax. He-normal init [7]. Nadam (lr=0.002). Batch=512, max 10 epochs, EarlyStopping(patience=3).
1. **Results**: Achieves strong macro F1 within the 60s runtime limit, significantly beating the baseline.


<font color=green><h3><b>$\epsilon$. References</b></h3>
<details>
  <summary>Instructions</summary>
  <div>

1. Cite your sources to help your peers learn from these (and to avoid plagiarism).
1. HOML textbook should be cited, since we used it in this week's learning.
1. Use Google Scholar to draw [APA](https://en.wikipedia.org/wiki/American_Psychological_Association) citation format for books and publications.
1. Cite [StackOverflow](https://stackoverflow.com/), YouTube videos, package docs, open-access textbooks/publicaitons and other meaningful internet resources that you used.
1. We may reward exceptional and meaningful citations (not just a list of [SKL](https://scikit-learn.org/stable/)/[TF](https://www.tensorflow.org/) manual pages and a list of articles.) For example, if you used an idea from a publication, indicate it in TGP with a number that corresponds to its reference in References.

</div> </details>
</font>

1. Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly Media.
2. Hastie, T., Tibshirani, R., & Friedman, J. (2017). *The Elements of Statistical Learning* (2nd ed.). Springer.
3. Wood, R. L., et al. (2020). Analysis of identification method for bacterial species using optical data from DNA oligomers. *Frontiers in Microbiology*, 11, 257.
4. Lin, T. Y., et al. (2017). Focal loss for dense object detection. *IEEE ICCV*, pp. 2980-2988.
5. Srivastava, N., et al. (2014). Dropout: A simple way to prevent neural networks from overfitting. *JMLR*, 15, pp. 1929-1958.
6. Smith, L. N. (2018). A disciplined approach to neural network hyper-parameters. *arXiv:1803.09820*.
7. Glorot, X. & Bengio, Y. (2010). Understanding the difficulty of training deep feedforward neural networks. *AISTATS*, pp. 249-256.
8. Abadi, M., et al. (2016). TensorFlow: A system for large-scale machine learning. *OSDI 16*, pp. 265-283.
9. Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *JMLR*, 12, pp. 2825-2830.
10. Ioffe, S. & Szegedy, C. (2015). Batch normalization: Accelerating deep network training. *ICML*, pp. 448-456.
11. Past competition summaries from SP25 and FA24 cohorts (notebook starter ideas).
12. Leo, C. (2024). The math behind NaDam Optimizer. *Towards Data Science*.
13. ReduceLROnPlateau. TensorFlow documentation. https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ReduceLROnPlateau
14. Dozat, T. (2016). Incorporating Nesterov Momentum into Adam. *ICLR Workshop*.


<hr color=green size=40><strong><font color=green size=5>⌛End of TGP. Do not exceed RTL! Do not write code outside TGP.</font></strong>
<!--<hr color=green size=40>-->

In [ ]:
tmr.ShowTime()    # measure Colab's runtime. Do not remove. Keep as the last cell in your notebook.

Runtime is 30 sec


<details>
  <summary><font size=5><b>💡Starter Ideas</b></font></summary>
  <div>
  
Try [**Common Starter Ideas**](https://colab.research.google.com/drive/1riOGrE_Fv-yfIbM5V4pgJx4DWcd92cZr#scrollTo=oD-fFbF2ZSBl&line=1&uniqifier=1)

### **Summarized ideas from past participants**
1. F₁=0.9917271826, rank=1, SP25:
    1. **Preprocessing:** We split the dataset into training and test sets to maintain an unbiased evaluation. They applied a deviation spectrum transformation to the data, which improved model robustness and performance, especially under noisy conditions. Standard scaling was performed to normalize features, followed by applying PCA to reduce dimensionality and retain key information. This combination of transformations maximized the information captured while making the model less sensitive to outliers and noise.
    1. **Modeling:** To address class imbalance (with rare classes comprising only about 5% of the data), we implemented a focal loss function, which emphasizes misclassified examples during training, improving rare class detection. They built a DNN with relu-activated dense layers, batch normalization, and dropout for regularization. The model was compiled using the Adam optimizer and evaluated using the macro F₁ score to fairly assess all classes. Hyperparameters, including those in the focal loss, network architecture, and training schedule, were tuned via random search for optimal results.
1. F₁=0.9905164735, rank=2, SP25:
    1. **Preprocessing:** We removed duplicate rows and filled missing values with the column means, ensuring a clean and consistent dataset. All features are converted to `float32` type to standardize data types and facilitate DNN training. The processed data then undergoes feature scaling and dimensionality reduction with PCA, retaining 99% of variance to reduce noise while preserving relevant information. This careful combination of imputation, normalization, and PCA improves model efficiency and consistency.
    1. **Modeling:** DNN with 5+ layers demonstrated superior performance. Extensive experimentation revealed that depth outperformed width, especially when combined with leaky ReLU activations. The best results were achieved by tuning hyperparameters, including layer count and activation types, resulting in an architecture with 4 layers and high accuracy (around 0.9972). Time and computational constraints guided the decision to optimize structure depth over width for the best balance of performance and efficiency.
1. F₁=0.9893170845, rank=3, SP25:
    1. **Preprocessing:** standardized features using `StandardScaler`, and dimensionality was reduced via PCA (retaining 99% variance). To address class imbalance, minority classes were oversampled by 500%, and the dataset was further bootstrapped with additional noise to encourage generalization. However, after experimentation, all preprocessing was ultimately discarded, favoring raw data input to let the model learn inherent patterns without imposed transformations.
    1. **Modeling:** Initial modeling explored complex DNN architectures, custom data generators, and tailored callbacks to maximize performance, including using custom classes to capture subtle patterns. Despite increased computational complexity, these advanced strategies did not yield decisive improvements. We opted for a simplified, computationally efficient modeling workflow, prioritizing direct training on raw data, which led to strong model performance while minimizing unnecessary complexity. This balance between experimentation and simplification informed the final decision-making process.
1. F₁=0.9933124647, rank=2, FA24:
    1. **Preprocessing:** We addressed skewed empirical probability features by applying a `log1p` transform to reduce skewness and stabilize DNN learning. To tackle the high dimensionality and potential noise from augmented data, they used PCA, retaining 99.7% of data variance. These steps aimed to improve generalization and efficiency, though they noted that log transformation could lead to overfitting, while PCA robustly reduced overfitting and sped up model training.
    1. **Modeling:** DNN architecture was built with multiple dense layers, batch normalization, and dropout for regularization. They incorporated a dynamic learning rate scheduler (ReduceLROnPlateau) and used the Nadam optimizer for adaptive gradient updates. The design choices, including dropouts and normalization, targeted overfitting and aimed for smoother convergence, demonstrating improved macro F₁-scores and training stability with preprocessing. Decision reasoning consistently focused on balancing model capacity with generalization and stable training dynamics.
1. F₁=0.9920596388, rank=3, FA24:
    1. **Preprocessing:** To address significant class imbalance in the dataset, we explored synthetic data generation techniques—specifically SMOTE and ADASYN—to improve minority class representation. Although both methods increased minority class recall, SMOTE was preferred for its superior recall rates and faster computation time. However, the additional runtime overhead kept these methods from full integration in the final pipeline. Additionally, dimensionality reduction via PCA was applied after standard scaling to retain 95% variance using fewer features, resulting in significant speedup with minimal accuracy loss.
    1. **Modeling :** The modeling approach emphasized optimizing the learning rate schedule, selecting an exponential-based piecewise constant strategy after comparing it with constant and 1-cycle schedules. This approach allowed rapid convergence, achieving high validation accuracy early in training. Further, exploration of feature engineering techniques such as Adaptive Cluster Lasso did not yield accuracy improvements beyond using PCA-reduced features. Overall, decisions were driven by the need for both computational efficiency and reliably high classification performance.


### **Further resources:**
1. Lin, T. Y., et al. *Focal loss for dense object detection*. In Proceedings of the IEEE ICCV (pp. 2980-2988). [2017](https://openaccess.thecvf.com/content_ICCV_2017/papers/Lin_Focal_Loss_for_ICCV_2017_paper.pdf)
1. Trivedi, S. *Understanding focal loss: A quick read*. Medium. [2019, April](https://medium.com/visionwizard/understanding-focal-loss-a-quick-read-b914422913e7)
1. Wood, R. L., et al. *Analysis of identification method for bacterial species and antibiotic resistance genes using optical data from DNA oligomers*. Frontiers in Microbiology, 11, 257. [2020](https://doi.org/10.3389/fmicb.2020.00257)
1. Mavrin, A. focal-loss: TensorFlow implementation of focal loss. GitHub. [2022](https://github.com/artemmavrin/focal-loss)
1. Nitish Srivastava et al., *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*. Journal of Machine Learning Research 15. pp. 1929–1958. 2014
1. Leslie N. Smith, *A Disciplined Approach to Neural Network Hyper-Parameters: Part 1—Learning Rate, Batch Size, Momentum, and Weight Decay*. [2018](https://arxiv.org/pdf/1803.09820)
1. Xavier Glorot and Yoshua Bengio, *Understanding the Difficulty of Training Deep Feedforward Neural Networks*, Proceedings of the 13th International Conference on Artificial Intelligence and Statistics (2010): 249–256
1. Abadi, M., et al. *TensorFlow: A system for large-scale machine learning*. 12th USENIX Symposium on Operating Systems Design and Implementation (OSDI 16), 265–283. [2016](https://arxiv.org/pdf/1603.04467)
1. Pedregosa, F., et al. *Scikit-learn: Machine learning in Python*. Journal of Machine Learning Research, 12. pp. 2825–2830. [2011](https://jmlr.csail.mit.edu/papers/volume12/pedregosa11a/pedregosa11a.pdf)
1. Ioffe, S., & Szegedy, C. *Batch normalization: Accelerating deep network training by reducing internal covariate shift*. Proceedings of the 32nd ICML, pp. 448–456. [2015](https://proceedings.mlr.press/v37/ioffe15.html)
1. `ReduceLROnPlateau`. PyTorch 2.4 documentation. [n.d.](https://pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.ReduceLROnPlateau.html)
1. Lau, S. *Learning rate schedules and adaptive learning rate methods for DL*. Medium. [6/20/2018](https://towardsdatascience.com/learning-rate-schedules-and-adaptive-learning-rate-methods-for-deep-learning-2c8f433990d1)
1. Devansh. *How does Batch Size impact your model learning*. Medium. [5/5/23](https://medium.com/geekculture/how-does-batch-size-impact-your-model-learning-2dd34d9fb1fa)
1. Leo, C. *The math behind NaDam Optimizer*. Towards Data Science. Medium. [5/18/24](https://towardsdatascience.com/the-math-behind-nadam-optimizer-47dc1970d2cc)
1. Rahman, M. What You Need to Know about Sparse Categorical Cross Entropy. Medium. [5/29/24](https://rmoklesur.medium.com/what-you-need-to-know-about-sparse-categorical-cross-entropy-9f07497e3a6f)
1. Chawla, N. V., et al. *SMOTE: synthetic minority over-sampling technique*. Journal of AI research, 16, 321-357. [2002](https://doi.org/10.1613/jair.953)
1. Blagus, R., & Lusa, L. *SMOTE for high-dimensional class-imbalanced data*. BMC bioinformatics, 14, 1-16. [2013](https://doi.org/10.1186/1471-2105-14-106)
1. He, H., et al. *ADASYN: Adaptive synthetic sampling approach for imbalanced learning*. In 2008 IEEE international joint conference on neural networks. IEEE world congress on computational intelligence. pp. 1322-1328. [2008(]https://doi.org/10.1109/IJCNN.2008.4633969)
1. Moeckel, C., et al. *A survey of k-mer methods and applications in bioinformatics. Computational and Structural Biotechnology Journal. [2024](https://doi.org/10.1016/j.csbj.2024.05.025)
1. Jaillard, M., et al. *Interpreting k-mer-based signatures for antibiotic resistance prediction*, GigaScience, v9, i10, [10/2020](https://doi.org/10.1093/gigascience/giaa110)
1. Ren, R., Yin, C., & S.-T. Yau, S. *kmer2vec: A novel method for comparing DNA sequences by word2vec embedding*. Journal of Computational Biology, 29(9), pp. 1001-1021. [2022](https://doi.org/10.1089/cmb.2021.0536)
1. Nguyen, T.T.D., et al. *Using k-mer embeddings learned from a Skip-gram based neural network for building a cross-species DNA N6-methyladenine site prediction model*. Plant Mol Biol 107, 533-542 [2021](https://doi.org/10.1007/s11103-021-01204-1)
1. Mejía-Guerra, M. K., & Buckler, E. S. *A k-mer grammar analysis to uncover maize regulatory architecture*. BMC plant biology, 19, 1-17. [2019](https://doi.org/10.1186/s12870-019-1693-2)
1. Vujovic, T. *Continuous Optimization with Piecewise-Decaying Learning Rate Scheduling for Medical Image Registration*. Journal of Computational Vision and Imaging Systems, 8(1), 78-80. [2022](https://doi.org/10.15353/jcvis.v8i1.5384)
1. E. M. Dogo, et al. *A Comparative Analysis of Gradient Descent-Based Optimization Algorithms on Convolutional Neural Networks*. 2018 International Conference on Computational Techniques, Electronics and Mechanical Systems (CTEMS), Belgaum, India, pp. 92-99, [2018](https://doi.org/10.1109/CTEMS.2018.8769211)
1. Takase, T., et al. *Why Does Large Batch Training Result in Poor Generalization? A Comprehensive Explanation and a Better Strategy from the Viewpoint of Stochastic Optimization*. Neural Comput; 30 (7): 2005-2023. [2018](https://doi.org/10.1162/neco_a_01089)
1. Smith, S. L., & Le, Q. V. *A bayesian perspective on generalization and stochastic gradient descent*. [2017](https://doi.org/10.48550/arXiv.1710.06451)
1. Opitz, D., & Shavlik, J. *Generating accurate and diverse members of a neural-network ensemble*. Advances in neural information processing systems, 8. [1995](https://proceedings.neurips.cc/paper_files/paper/1995/hash/1896a3bf730516dd643ba67b4c447d36-Abstract.html)
1. Bengio, Y. *Practical recommendations for gradient-based training of deep architectures*. In Neural networks: Tricks of the trade: Second edition (pp. 437-478). [2012](https://doi.org/10.48550/arXiv.1206.5533)

</div> </details>
